In [1]:
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.size": 12,
})

textwidth_pt = 472
inch_per_pt = 1 / 72.27
fig_width = textwidth_pt * inch_per_pt
fig_height = fig_width * 0.65

In [2]:
import warnings
warnings.filterwarnings('ignore')

import copy
import itertools
import pypesto
import petab
import fides
import pypesto.petab
import pypesto.optimize as optimize
import pypesto.visualize as visualize
import pypesto.visualize.model_fit
import amici
import numpy as np
import numpy.matlib as nm
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import libsbml
import sys
from pypesto.store import save_to_hdf5
import dill
import pypesto.hierarchical
from pypesto.store import read_from_hdf5
import os
from pathlib import Path

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "petab").is_dir() and (path / "results").is_dir() and (path / "src").is_dir()
)
os.chdir(PROJECT_ROOT)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
PETAB_DIR = PROJECT_ROOT / "petab"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
MODEL_SELECTION_RESULTS_DIR = RESULTS_DIR / "model_selection" / "all_ligands_sacess_brute_force_mixed_noise"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
(MODEL_SELECTION_RESULTS_DIR / "figures").mkdir(parents=True, exist_ok=True)
(MODEL_SELECTION_RESULTS_DIR / "figures" / "clustering").mkdir(parents=True, exist_ok=True)

import src.functionalities as func
import src.model_selection_functionalities as msfunc

In [3]:
import importlib
importlib.reload(func)
importlib.reload(msfunc)

from matplotlib.pyplot import cm

In [4]:
import petab_select
from petab_select.constants import PETAB_PROBLEM
from pypesto.select import model_to_pypesto_problem

In [5]:
def objective_customizer(objective):
    objective.amici_solver.setMaxSteps(100000)
    objective.amici_solver.setRelativeTolerance(1e-12)
    objective.amici_solver.setAbsoluteTolerance(1e-8)
    objective.amici_solver.setSensitivityMethod(amici.SensitivityMethod_adjoint)
    return None

In [11]:
os.getcwd()
output_path = Path(os.path.join(os.getcwd(), "revision_test/results/model_selection/"))

petab_select_yaml = Path(os.getcwd()) / 'revision_test' /  'petab_select' / 'petab_select_problem.yaml'
petab_select_problem = petab_select.Problem.from_yaml(petab_select_yaml)

In [13]:
sbml_file = Path(os.getcwd()) / 'revision_test' /  'petab_select' / 'AhR_v3_diploid.xml'

In [14]:
sbml_importer = amici.SbmlImporter(
    sbml_file,
    show_sbml_warnings=True,
)

sbml_importer.sbml2amici(
    "model_audit",
    "model_audit",
    compute_conservation_laws=False,
)

2026-10-13 11:11:47.910 - amici.sbml_import - DEBUG - Finished validating SBML                     + (9.34E-03s)
2026-10-13 11:11:47.923 - amici.sbml_import - DEBUG - Finished converting SBML functions           + (5.69E-03s)
2026-10-13 11:11:47.930 - amici.sbml_import - DEBUG - Finished converting SBML local parameters    + (3.90E-05s)
2026-10-13 11:11:47.930 - amici.sbml_import - INFO - Finished loading SBML                           (3.27E-02s)
2026-10-13 11:11:47.931 - amici.sbml_import - DEBUG - Changing log_level from 10 to 40
2026-10-13 11:11:48.121 - amici.de_model - DEBUG - Changing log_level from 10 to 40
2026-10-13 11:11:48.220 - amici.de_export - DEBUG - Changing log_level from 10 to 40


In [12]:
petab_yaml = Path(os.getcwd()) / 'revision_test' /  'petab_select' / 'full_petab_problem.yaml'

selection_petab_problem = petab.Problem.from_yaml(petab_yaml)

selection_pypesto_importer = pypesto.petab.PetabImporter(selection_petab_problem, hierarchical=False)
selection_model = selection_pypesto_importer.create_model(force_compile=True)

Visualization table not available. Skipping.
Compiling amici model to folder /home/vincent/PhD/Projects/Ahr_modelling/Mathematical-Modeling-of-the-Canonical-Aryl-Hydrocarbon-Receptor-Pathway/amici_models/0.30.0/AhR_canonical_v3_diploid.
2026-10-13 11:01:01.617 - amici.petab.sbml_import - INFO - Importing model ...
2026-10-13 11:01:01.617 - amici.petab.sbml_import - INFO - Validating PEtab problem ...
Visualization table not available. Skipping.
2026-10-13 11:01:01.646 - amici.petab.sbml_import - INFO - Model name is 'AhR_canonical_v3_diploid'.
Writing model code to '/home/vincent/PhD/Projects/Ahr_modelling/Mathematical-Modeling-of-the-Canonical-Aryl-Hydrocarbon-Receptor-Pathway/amici_models/0.30.0/AhR_canonical_v3_diploid'.
2026-10-13 11:01:01.648 - amici.petab.sbml_import - INFO - Species: 18
2026-10-13 11:01:01.648 - amici.petab.sbml_import - INFO - Global parameters: 54
2026-10-13 11:01:01.649 - amici.petab.sbml_import - INFO - Reactions: 24
2026-10-13 11:01:01.686 - amici.petab.sbm

running build_ext
running AmiciBuildCMakeExtension
------------------------------ model_ext ------------------------------

==> Configuring:
$ cmake -S /home/vincent/PhD/Projects/Ahr_modelling/Mathematical-Modeling-of-the-Canonical-Aryl-Hydrocarbon-Receptor-Pathway/amici_models/0.30.0/AhR_canonical_v3_diploid -B /home/vincent/PhD/Projects/Ahr_modelling/Mathematical-Modeling-of-the-Canonical-Aryl-Hydrocarbon-Receptor-Pathway/amici_models/0.30.0/AhR_canonical_v3_diploid/build_model_ext -G Ninja -DCMAKE_MAKE_PROGRAM=/home/vincent/PhD/Projects/Ahr_modelling/Mathematical-Modeling-of-the-Canonical-Aryl-Hydrocarbon-Receptor-Pathway/ahr_env/bin/ninja -DCMAKE_BUILD_TYPE=Release -DCMAKE_INSTALL_PREFIX:PATH=/home/vincent/PhD/Projects/Ahr_modelling/Mathematical-Modeling-of-the-Canonical-Aryl-Hydrocarbon-Receptor-Pathway/amici_models/0.30.0/AhR_canonical_v3_diploid/AhR_canonical_v3_diploid -DCMAKE_VERBOSE_MAKEFILE=ON -DCMAKE_PREFIX_PATH=/home/vincent/PhD/Projects/Ahr_modelling/Mathematical-Modeling